# UFA Data Cleaning Functions

This notebook contains key data cleaning and processing functions for UFA game data:
- `clean_data`: Processes and zips home/away events into chronological order
- `insert_missing_turnovers`: Post-processes events to add synthetic turnovers where missing
- `average_turnover_coordinates`: Averages turnover coordinate mismatches between teams

## Imports

In [1]:
import requests
import pandas as pd
import numpy as np
import random
from typing import List, Dict, Optional, Tuple

## API Data Retrieval

Functions to fetch metadata from the UFA Stats API.

In [2]:
def get_teams_data(years="all"):
    """
    Fetches team data from the UFA Stats API.
    
    Args:
        years: A comma-delimited list of four digit years or "all" to include all years
        
    Returns:
        List of team dictionaries with keys: team_id, team_name, division
    """
    url = "https://www.backend.ufastats.com/api/v1/teams"
    params = {'years': years}
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json().get("data", [])
        
        teams = []
        for team in data:
            teams.append({
                'team_id': team.get('teamID'),
                'team_name': team.get('fullName'),
                'division': team.get('division', {}).get('name'),
                'year': team.get('year')
            })
        
        return teams
    
    except requests.exceptions.RequestException as e:
        print(f"Error fetching teams data: {e}")
        return []

In [3]:
def get_games(date=None, game_ids=None, statuses="Final", team_ids=None):
    """
    Fetches games from the UFA Stats API.
    
    Args:
        date: Date range (e.g., "2024", "2024-04:2024-05", "2024-04-27:")
        game_ids: Comma-delimited list of game IDs (alternative to date)
        statuses: Comma-delimited list of statuses (default: "Final")
        team_ids: Optional comma-delimited list of team IDs
        
    Returns:
        List of game dictionaries with keys: game_id, home_team_id, away_team_id, 
                                             home_score, away_score, year, game_date
    """
    url = "https://www.backend.ufastats.com/api/v1/games"
    
    params = {}
    if date:
        params['date'] = date
    elif game_ids:
        params['gameIDs'] = game_ids
    else:
        raise ValueError("Must provide either 'date' or 'game_ids' parameter")
    
    if statuses:
        params['statuses'] = statuses
    if team_ids:
        params['teamIDs'] = team_ids
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json().get("data", [])
        
        games = []
        for game in data:
            game_id = game.get('gameID')
            # Extract year from gameID (format: YYYY-MM-DD-TEAM-TEAM)
            year = int(game_id[:4]) if game_id else None
            game_date = game_id[:10] if game_id else None  # YYYY-MM-DD
            
            games.append({
                'game_id': game_id,
                'home_team_id': game.get('homeTeamID'),
                'away_team_id': game.get('awayTeamID'),
                'home_score': game.get('homeScore'),
                'away_score': game.get('awayScore'),
                'year': year,
                'game_date': game_date,
                'status': game.get('status')
            })
        
        return games
    
    except requests.exceptions.RequestException as e:
        print(f"Error fetching games data: {e}")
        return []

In [4]:
def get_players_data(years="all", team_ids=None):
    """
    Fetches player data from the UFA Stats API.
    
    Args:
        years: A comma-delimited list of four digit years or "all" to include all years
        team_ids: Optional comma-delimited list of team IDs
        
    Returns:
        List of player dictionaries with keys: player_id, full_name, team_id, year
        Note: Returns one entry per player per team per year
    """
    url = "https://www.backend.ufastats.com/api/v1/players"
    
    params = {'years': years}
    if team_ids:
        params['teamIDs'] = team_ids
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json().get("data", [])
        
        players = []
        for player in data:
            player_id = player.get('playerID')
            first_name = player.get('firstName', '')
            last_name = player.get('lastName', '')
            full_name = f"{first_name} {last_name}".strip()
            
            # Create an entry for each team the player played for
            teams = player.get('teams', [])
            if teams:
                for team in teams:
                    players.append({
                        'player_id': player_id,
                        'full_name': full_name,
                        'team_id': team.get('teamID'),
                        'year': team.get('year'),
                        'active': team.get('active', True),
                        'jersey_number': team.get('jerseyNumber')
                    })
            else:
                # Player with no teams
                players.append({
                    'player_id': player_id,
                    'full_name': full_name,
                    'team_id': None,
                    'year': None,
                    'active': None,
                    'jersey_number': None
                })
        
        return players
    
    except requests.exceptions.RequestException as e:
        print(f"Error fetching players data: {e}")
        return []

### Example: Fetching API Data

In [5]:
# Example: Fetch teams for 2024
teams_2024 = get_teams_data(years="2024")
print(f"Found {len(teams_2024)} teams")
print(f"Sample: {teams_2024[0] if teams_2024 else 'No data'}")

# Example: Fetch games for April 2024
games_april = get_games(date="2024-04")
print(f"\nFound {len(games_april)} games in April 2024")
print(f"Sample: {games_april[0] if games_april else 'No data'}")

# Example: Fetch players for 2024
players_2024 = get_players_data(years="2024")
print(f"\nFound {len(players_2024)} player-team records for 2024")
print(f"Sample: {players_2024[0] if players_2024 else 'No data'}")

Found 24 teams
Sample: {'team_id': 'aviators', 'team_name': 'LA Aviators', 'division': 'West', 'year': 2024}

Found 9 games in April 2024
Sample: {'game_id': '2024-04-27-MIN-PIT', 'home_team_id': 'thunderbirds', 'away_team_id': 'windchill', 'home_score': 11, 'away_score': 18, 'year': 2024, 'game_date': '2024-04-27', 'status': 'Final'}

Found 870 player-team records for 2024
Sample: {'player_id': 'mkiyoi', 'full_name': 'Michael Kiyoi', 'team_id': 'aviators', 'year': 2024, 'active': True, 'jersey_number': '1'}


## Helper Functions

In [6]:
# Field bounds for coordinate validation
FIELD_X_MIN, FIELD_X_MAX = -25.0, 25.0
FIELD_Y_MIN, FIELD_Y_MAX = 0.0, 110.0

def _clamp(v: float, lo: float, hi: float) -> float:
    """Clamp a value between lo and hi."""
    return max(lo, min(hi, v))

def _jitter_endzone_xy(
    rng: random.Random,
    base_x: float = 0.0,
    base_y: float = 110.0,
    max_dx: float = 5.0,
    max_dy: float = 15.0,
) -> Tuple[float, float]:
    """
    Pick a turnover location near back-center of the end zone, with bounded jitter,
    then clamp to field bounds.
    """
    x = base_x + rng.uniform(-max_dx, max_dx)
    y = base_y + rng.uniform(-max_dy, max_dy)
    return float(_clamp(x, FIELD_X_MIN, FIELD_X_MAX)), float(_clamp(y, FIELD_Y_MIN, FIELD_Y_MAX))

## Main Functions

### 2. clean_data

Processes raw game event data by zipping home and away team events into chronological order.

In [7]:
def clean_data(game_ids):
    """
    Fetches and cleans game events by merging home and away events into chronological order.
    
    Args:
        game_ids: List of game IDs to process
        
    Returns:
        dict: Dictionary mapping gameID to list of combined events
    """
    events_url = 'https://www.backend.ufastats.com/api/v1/gameEvents'
    games_url  = 'https://www.backend.ufastats.com/api/v1/games'
    
    # Get game details.
    game_ids_str = ','.join(game_ids)
    games_params = {'gameIDs': game_ids_str}
    try:
        games_response = requests.get(games_url, params=games_params)
        games_response.raise_for_status()
        games_data = games_response.json().get("data", [])
        team_id_map = {}
        for game in games_data:
            year = game["gameID"][:4]
            team_id_map[game["gameID"]] = {
                "home": game["homeTeamID"],
                "away": game["awayTeamID"],
                "year": year
            }
    except requests.exceptions.RequestException as e:
        print(f"Error fetching game data from {games_url}: {e}")
        return None
    
    # Dictionary to hold the zipped/combined events for each game.
    results = {}
    
    # Process each game separately.
    for gameID in game_ids:
        params = {'gameID': gameID}
        try:
            response = requests.get(events_url, params=params)
            if response.status_code == 200 and response.headers.get('Content-Type', '').startswith('application/json'):
                data = response.json()
                # Get team IDs and game year.
                home_team_id = team_id_map.get(gameID, {}).get("home", "Unknown")
                away_team_id = team_id_map.get(gameID, {}).get("away", "Unknown")
                game_year    = int(team_id_map.get(gameID, {}).get("year", "0"))
                
                home_events = data.get("data", {}).get("homeEvents", [])
                away_events = data.get("data", {}).get("awayEvents", [])
                
                # Tag each event with team, year, and gameID.
                for event in home_events:
                    event['team'] = home_team_id
                    event['year'] = game_year
                    event['gameID'] = gameID
                for event in away_events:
                    event['team'] = away_team_id
                    event['year'] = game_year
                    event['gameID'] = gameID
                
                homeIndex = 0
                awayIndex = 0
                homeStack = []
                awayStack = []
                homeStackIndex = 0
                awayStackIndex = 0
                switch = 0  # Used to switch between home and away for zipping.
                combinedStack = []
                
                while homeIndex < len(home_events) or awayIndex < len(away_events):
                    homeStack = []
                    awayStack = []
                    # Build homeStack: Append events until (and including) a termination event.
                    while homeIndex < len(home_events):
                        homeStack.append(home_events[homeIndex])
                        if home_events[homeIndex]['type'] in {15, 19, 28, 29, 30, 31, 32, 33}:
                            homeIndex += 1
                            break
                        homeIndex += 1

                    # Build awayStack similarly.
                    while awayIndex < len(away_events):
                        awayStack.append(away_events[awayIndex])
                        if away_events[awayIndex]['type'] in {15, 19, 28, 29, 30, 31, 32, 33}:
                            awayIndex += 1
                            break
                        awayIndex += 1

                    # Set up header of the combined stack.
                    if homeStack and homeStack[0]['type'] == 1:
                        combinedStack.append(homeStack[0])
                        if awayStack:
                            combinedStack.append(awayStack[0])
                        if len(homeStack) > 1:
                            combinedStack.append(homeStack[1])
                        homeStackIndex = 2
                        awayStackIndex = 1
                        switch = 1
                    else:
                        if awayStack:
                            combinedStack.append(awayStack[0])
                        if homeStack:
                            combinedStack.append(homeStack[0])
                        if len(awayStack) > 1:
                            combinedStack.append(awayStack[1])
                        homeStackIndex = 1
                        awayStackIndex = 2
                        switch = 0

                    while homeStackIndex < len(homeStack) or awayStackIndex < len(awayStack):
                        if switch == 0 and homeStackIndex < len(homeStack):
                            while homeStackIndex < len(homeStack):
                                event = homeStack[homeStackIndex]
                                combinedStack.append(event)
                                homeStackIndex += 1
                                if event['type'] in {3, 5, 15, 19, 20, 22, 23, 24, 28, 29, 30, 31, 32, 33}:
                                    switch = 1
                                    break
                                elif event['type'] == 17:
                                    # BOUNDS CHECK ADDED HERE
                                    if awayStackIndex < len(awayStack) and awayStack[awayStackIndex]['type'] == 16:
                                        combinedStack.append(awayStack[awayStackIndex])
                                        awayStackIndex += 1
                                    else:
                                        combinedStack.append({'type': 16, 'team': home_team_id, 'year': game_year, 'gameID': gameID})
                                elif event['type'] == 16:
                                    # BOUNDS CHECK ADDED HERE
                                    if awayStackIndex < len(awayStack) and awayStack[awayStackIndex]['type'] == 17:
                                        combinedStack.insert(-1, awayStack[awayStackIndex])
                                        awayStackIndex += 1
                                    else:
                                        combinedStack.insert(-1, {'type': 17, 'team': away_team_id, 'year': game_year, 'gameID': gameID})
                                    break
                                        
                        elif switch == 1 and awayStackIndex < len(awayStack):
                            while awayStackIndex < len(awayStack):
                                event = awayStack[awayStackIndex]
                                combinedStack.append(event)
                                awayStackIndex += 1
                                if event['type'] in {3, 5, 15, 19, 20, 22, 23, 24, 28, 29, 30, 31, 32, 33}:
                                    switch = 0
                                    break
                                elif event['type'] == 17:
                                    # BOUNDS CHECK ADDED HERE
                                    if homeStackIndex < len(homeStack) and homeStack[homeStackIndex]['type'] == 16:
                                        combinedStack.append(homeStack[homeStackIndex])
                                        homeStackIndex += 1
                                    else:
                                        combinedStack.append({'type': 16, 'team': home_team_id, 'year': game_year, 'gameID': gameID})
                                elif event['type'] == 16:
                                    # BOUNDS CHECK ADDED HERE
                                    if homeStackIndex < len(homeStack) and homeStack[homeStackIndex]['type'] == 17:
                                        combinedStack.insert(-1, homeStack[homeStackIndex])
                                        homeStackIndex += 1
                                    else:
                                        combinedStack.insert(-1, {'type': 17, 'team': home_team_id, 'year': game_year, 'gameID': gameID})
                                    break
                        else:
                            break

                # Store the zipped/combined result for this game.
                results[gameID] = combinedStack
            else:
                print(f"Unexpected response for game {gameID}: {response.status_code}")
        except requests.exceptions.RequestException as e:
            print(f"Error fetching event data from {events_url} for game {gameID}: {e}")
    
    return results

### 3. insert_missing_turnovers

Post-processes combined event lists to insert synthetic turnovers where they're missing from the data.

In [8]:
def insert_missing_turnovers(
    combined_events: List[Dict],
    *,
    seed: Optional[int] = None
) -> List[Dict]:
    """
    Post-process a single game's combined event list and insert synthetic turnovers (type 22)
    when missing between a completed pass (type 18) and the next possession delimiter (type 28).

    Also enforces ordering so that defender credits (type 11) appear BEFORE type 28.

    Rule:
      - After a type 18, scan forward through any number of type 11 (defender credit) events.
      - If we encounter type 22 before type 28: do nothing.
      - If we encounter type 28 before type 22: insert a synthetic type 22 RIGHT BEFORE that 28.
      - Independently, ensure any '11' that would follow a '28' is moved before it.

    Returns
    -------
    List[Dict]
        New list with any needed type 22 events inserted and 11/28 order corrected.
        Synthetic events will have a "synthetic": True field.
        Synthetic turnover coordinates are rounded to 2 decimal places.
    """
    rng = random.Random(seed)
    out: List[Dict] = []
    i = 0
    n = len(combined_events)

    while i < n:
        e = combined_events[i]
        out.append(e)

        # Ensure 11 always precedes 28 if they were adjacent in the source
        # (bubble the just-appended 11 leftwards across consecutive 28s).
        k = len(out) - 1
        while k > 0 and out[k].get("type") == 11 and out[k-1].get("type") == 28:
            out[k-1], out[k] = out[k], out[k-1]
            k -= 1

        if e.get("type") == 18:
            # Look ahead to find the next decisive event (22 or 28), allowing any number of 11s
            j = i + 1
            saw_22 = False
            insert_pos_in_out: Optional[int] = None

            while j < n:
                t = combined_events[j].get("type")
                if t == 22:
                    saw_22 = True
                    break
                if t == 28:
                    # Insert just before this 28 in the output stream
                    insert_pos_in_out = len(out)
                    break
                if t not in (11,):  # tolerate 11s; stop on other unexpected stuff
                    break
                j += 1

            if (not saw_22) and (insert_pos_in_out is not None):
                # Build synthetic 22 using the receiver of the 18
                thrower = e.get("receiver")
                if thrower is not None:
                    tX, tY = _jitter_endzone_xy(rng)
                    synth_22 = {
                        "type": 22,
                        "thrower": thrower,
                        "throwerX": e.get("receiverX"),
                        "throwerY": e.get("receiverY"),
                        "turnoverX": round(tX, 2),
                        "turnoverY": round(tY, 2),
                        "team": e.get("team"),
                        "year": e.get("year"),
                        "gameID": e.get("gameID"),
                        "synthetic": True  # Mark as synthetic
                    }
                    out.insert(insert_pos_in_out, synth_22)

        i += 1

    return out

### 4b. average_turnover_coordinates

Averages turnover coordinate mismatches between teams. Since `normalize_coordinates()` has already
put all coordinates in the same physical frame, we can average directly without flipping.

### 4a. normalize_coordinates

Flips away team coordinates to home team's physical frame so all events share the same coordinate system.
After normalization: home team attacks toward Y≈110, away team attacks toward Y≈10.

In [9]:
def normalize_coordinates(combined_events: List[Dict], away_team_id: str) -> List[Dict]:
    """
    Flip away team coordinates to home team's physical frame.
    
    The UFA API returns events where both teams record in their own frame
    (attacking toward Y≈110). This means for the same physical point:
    - Home team records (x, y)
    - Away team records (-x, 120-y)
    
    After this function: home attacks toward Y=110, away attacks toward Y=10.
    All coordinates are in the same physical frame.
    
    Parameters
    ----------
    combined_events : List[Dict]
        Event list from clean_data + insert_missing_turnovers
    away_team_id : str
        The away team's ID (events from this team get flipped)
    
    Returns
    -------
    List[Dict]
        Event list with away team coordinates flipped to home frame
    """
    COORD_FIELDS = [
        ('throwerX', 'throwerY'),
        ('receiverX', 'receiverY'),
        ('turnoverX', 'turnoverY'),
        ('pullX', 'pullY'),
    ]
    
    events = [event.copy() for event in combined_events]
    
    for event in events:
        if event.get('team') != away_team_id:
            continue
        for x_key, y_key in COORD_FIELDS:
            if event.get(x_key) is not None:
                event[x_key] = round(-event[x_key], 2)
            if event.get(y_key) is not None:
                event[y_key] = round(120 - event[y_key], 2)
    
    return events

In [10]:
def average_turnover_coordinates(
    combined_events: List[Dict]
) -> List[Dict]:
    """
    Averages turnover coordinate mismatches between teams.
    
    Assumes normalize_coordinates() has already been called, so all coordinates
    are in the same physical frame. We simply average the turnover end location
    with the next possession's start location.
    
    Rules:
    - For drops (type 20): use receiverX, receiverY as turnover end location
    - For throwaways (type 22): use turnoverX, turnoverY as turnover end location
    - Average directly (both coords are in the same physical frame)
    - Store the average in both events
    - SKIP averaging if turnover is in endzone (Y < 20 or Y > 100)
    - Round averaged coordinates to 2 decimal places
    
    Parameters
    ----------
    combined_events : List[Dict]
        Event list (already normalized to same physical frame)
    
    Returns
    -------
    List[Dict]
        Event list with averaged turnover coordinates
    """
    # Work on a copy to avoid modifying the original
    events = [event.copy() for event in combined_events]
    i = 0
    
    while i < len(events):
        event = events[i]
        event_type = event.get('type')
        
        # Check if this is a turnover event
        if event_type in (20, 22):
            turnover_team = event.get('team')
            
            # Get turnover end coordinates
            if event_type == 20:  # Drop - use receiver coordinates
                turnover_x = event.get('receiverX')
                turnover_y = event.get('receiverY')
                turnover_x_key = 'receiverX'
                turnover_y_key = 'receiverY'
            else:  # Throwaway (type 22) - use turnover coordinates
                turnover_x = event.get('turnoverX')
                turnover_y = event.get('turnoverY')
                turnover_x_key = 'turnoverX'
                turnover_y_key = 'turnoverY'
            
            # Only proceed if we have valid turnover coordinates
            if turnover_x is not None and turnover_y is not None:
                # Skip if in endzone (Y < 20 or Y > 100)
                if turnover_y < 20 or turnover_y > 100:
                    i += 1
                    continue
                
                # Find next event from opposing team with thrower coordinates
                j = i + 1
                while j < len(events):
                    next_event = events[j]
                    next_team = next_event.get('team')
                    next_thrower_x = next_event.get('throwerX')
                    next_thrower_y = next_event.get('throwerY')
                    
                    # Found the start of next possession
                    if (next_team != turnover_team and 
                        next_thrower_x is not None and 
                        next_thrower_y is not None):
                        
                        # Both coords are in the same physical frame now,
                        # so average directly
                        avg_x = round((turnover_x + next_thrower_x) / 2, 2)
                        avg_y = round((turnover_y + next_thrower_y) / 2, 2)
                        
                        # Store the average in both events
                        events[i][turnover_x_key] = avg_x
                        events[i][turnover_y_key] = avg_y
                        events[j]['throwerX'] = avg_x
                        events[j]['throwerY'] = avg_y
                        
                        break
                    
                    j += 1
        
        i += 1
    
    return events

## Example Usage

In [11]:
# Example 1: Clean data (zip home/away events)
cleaned_data = clean_data(["2024-04-27-MTL-NY"])

for game_id, events in cleaned_data.items():
    print(f"Game {game_id}: {len(events)} events")
    print(f"First 5 events: {events[:5]}")

Game 2024-04-27-MTL-NY: 750 events
First 5 events: [{'type': 1, 'line': ['cguay', 'swarrick', 'jbernat', 'rlalondel', 'tlalondel', 'jduquette', 'ctremblay'], 'time': 0, 'team': 'royal', 'year': 2024, 'gameID': '2024-04-27-MTL-NY'}, {'type': 2, 'line': ['lhaberfie', 'bjagt', 'skeegan', 'ochartock', 'srueschem', 'cweinberg', 'jwilliams'], 'time': 0, 'team': 'empire', 'year': 2024, 'gameID': '2024-04-27-MTL-NY'}, {'type': 7, 'puller': 'ctremblay', 'pullX': -16.47, 'pullY': 89.19, 'pullMs': 3152, 'team': 'royal', 'year': 2024, 'gameID': '2024-04-27-MTL-NY'}, {'type': 18, 'thrower': 'skeegan', 'throwerX': 22.64, 'throwerY': 7.09, 'receiver': 'srueschem', 'receiverX': 4.63, 'receiverY': 9.4, 'team': 'empire', 'year': 2024, 'gameID': '2024-04-27-MTL-NY'}, {'type': 18, 'thrower': 'srueschem', 'throwerX': 4.63, 'throwerY': 9.4, 'receiver': 'ochartock', 'receiverX': -13.51, 'receiverY': 18.61, 'team': 'empire', 'year': 2024, 'gameID': '2024-04-27-MTL-NY'}]


In [12]:
# Example 2: Insert missing turnovers
cleaned_data = clean_data(["2024-04-27-MTL-NY"])

for game_id, events in cleaned_data.items():
    processed_events = insert_missing_turnovers(events, seed=42)
    print(f"Game {game_id}:")
    print(f"  Original events: {len(events)}")
    print(f"  After processing: {len(processed_events)}")
    print(f"  Synthetic turnovers added: {len(processed_events) - len(events)}")

Game 2024-04-27-MTL-NY:
  Original events: 750
  After processing: 751
  Synthetic turnovers added: 1


In [13]:
# Example 3: Average turnover coordinates (with orientation correction)
cleaned_data = clean_data(["2024-04-27-MTL-NY"])

for game_id, events in cleaned_data.items():
    # Apply both processing steps
    fixed_events = insert_missing_turnovers(events, seed=42)
    averaged_events = average_turnover_coordinates(fixed_events)
    
    print(f"Game {game_id}:")
    print(f"  Original events: {len(events)}")
    print(f"  After insert_missing_turnovers: {len(fixed_events)}")
    print(f"  After average_turnover_coordinates: {len(averaged_events)}")
    
    # Find turnovers and show coordinate averaging WITH orientation correction
    print(f"\n  Turnover coordinate adjustments (with orientation flip):")
    print(f"  {'Type':<12} {'Y coord':<10} {'Status'}")
    print(f"  {'-'*40}")
    
    for i, event in enumerate(averaged_events):
        if event.get('type') in (20, 22):
            event_type_name = "Drop" if event.get('type') == 20 else "Throwaway"
            if event.get('type') == 20:
                y = event.get('receiverY')
            else:
                y = event.get('turnoverY')
            
            # Check if in endzone
            if y is None:
                status = "(no coords)"
            elif y < 20 or y > 100:
                status = "(endzone - not averaged)"
            else:
                status = "(orientation-corrected & averaged)"
            
            y_str = f"{y:.2f}" if y is not None else "N/A"
            print(f"  {event_type_name:<12} {y_str:<10} {status}")

Game 2024-04-27-MTL-NY:
  Original events: 750
  After insert_missing_turnovers: 751
  After average_turnover_coordinates: 751

  Turnover coordinate adjustments (with orientation flip):
  Type         Y coord    Status
  ----------------------------------------
  Throwaway    50.21      (orientation-corrected & averaged)
  Throwaway    62.95      (orientation-corrected & averaged)
  Throwaway    49.76      (orientation-corrected & averaged)
  Throwaway    110.75     (endzone - not averaged)
  Throwaway    54.60      (orientation-corrected & averaged)
  Throwaway    107.25     (endzone - not averaged)
  Throwaway    57.08      (orientation-corrected & averaged)
  Throwaway    101.72     (endzone - not averaged)
  Throwaway    55.22      (orientation-corrected & averaged)
  Drop         63.80      (orientation-corrected & averaged)
  Throwaway    76.64      (orientation-corrected & averaged)
  Throwaway    58.43      (orientation-corrected & averaged)
  Throwaway    104.93     (endzone 

## Complete Pipeline: Print All Fixed Events

In [14]:
# Complete pipeline: Clean data and insert missing turnovers
game_ids = ["2024-04-27-MTL-NY"]

# Step 1: Clean and merge home/away events
print("=" * 80)
print("STEP 1: Cleaning and merging home/away events")
print("=" * 80)
cleaned_data = clean_data(game_ids)

for game_id, events in cleaned_data.items():
    print(f"\nGame: {game_id}")
    print(f"Total events after cleaning: {len(events)}\n")
    
    # Step 2: Insert missing turnovers
    print("=" * 80)
    print("STEP 2: Inserting missing turnovers")
    print("=" * 80)
    fixed_events = insert_missing_turnovers(events, seed=42)
    
    print(f"\nOriginal events: {len(events)}")
    print(f"Fixed events: {len(fixed_events)}")
    print(f"Synthetic turnovers added: {len(fixed_events) - len(events)}\n")
    
    # Step 3: Print all fixed events with details
    print("=" * 80)
    print("STEP 3: All Fixed Events (Chronological Order)")
    print("=" * 80)
    
    # Event type mapping for readability
    event_types = {
        1: "Starting O",
        2: "Starting D",
        3: "Mid Point Timeout",
        4: "Between Point Timeout",
        5: "Mid Point Timeout",
        6: "Between Point Timeout",
        7: "Pull (Inbounds)",
        8: "Pull (OB)",
        9: "Offsides",
        10: "Offsides",
        11: "Defender Credit (Block)",
        12: "Callahan",
        13: "Switch Flag (Other Team Drop/Throwaway)",
        14: "Stall",
        15: "Score (Other Team)",
        16: "Penalty",
        17: "Penalty",
        18: "Pass",
        19: "Goal",
        20: "Drop",
        21: "Dropped Pull",
        22: "Throwaway",
        23: "Callahan",
        24: "Stall",
        25: "Injury",
        26: "Player Misconduct Foul",
        27: "Player Ejected",
        28: "End Q1",
        29: "End Q2",
        30: "End Q3",
        31: "End Q4",
        32: "End OT1",
        33: "End OT2"
    }
    
    # Get team IDs from first few events
    team_ids = set()
    for event in fixed_events[:10]:
        if 'team' in event:
            team_ids.add(event['team'])
    team_list = sorted(list(team_ids))
    
    # Initialize score tracking
    if len(team_list) >= 2:
        team1, team2 = team_list[0], team_list[1]
        score = {team1: 0, team2: 0}
    else:
        team1 = team_list[0] if team_list else "Unknown"
        team2 = "Unknown"
        score = {team1: 0, team2: 0}
    
    for idx, event in enumerate(fixed_events, 1):
        event_type = event.get('type', 'Unknown')
        event_name = event_types.get(event_type, f"Type {event_type}")
        team = event.get('team', 'N/A')
        
        # Update score if it's a goal
        if event_type == 19 and team in score:
            score[team] += 1
        
        # Check if event is synthetic
        is_synthetic = event.get('synthetic', False)
        synthetic_marker = " [SYNTHETIC]" if is_synthetic else ""
        
        # Format score display
        score_display = f"{team1}: {score[team1]} - {team2}: {score[team2]}"
        
        print(f"\n{idx}. {event_name}{synthetic_marker}")
        print(f"   Team: {team}")
        print(f"   Score: {score_display}")
        
        # Print relevant fields based on event type
        if event_type in [18, 19]:  # Throws and Goals
            print(f"   Thrower: {event.get('thrower', 'N/A')} ({event.get('throwerX', 'N/A')}, {event.get('throwerY', 'N/A')})")
            print(f"   Receiver: {event.get('receiver', 'N/A')} ({event.get('receiverX', 'N/A')}, {event.get('receiverY', 'N/A')})")
        elif event_type in [20, 22, 23]:  # Drops, Throwaways, Callahans
            print(f"   Thrower: {event.get('thrower', 'N/A')} ({event.get('throwerX', 'N/A')}, {event.get('throwerY', 'N/A')})")
            if 'turnoverX' in event:
                print(f"   Turnover Location: ({event.get('turnoverX', 'N/A')}, {event.get('turnoverY', 'N/A')})")
            if event_type == 20 and 'receiver' in event:
                print(f"   Receiver (dropped): {event.get('receiver', 'N/A')}")
        elif event_type == 11:  # Blocks
            print(f"   Defender: {event.get('defender', 'N/A')}")
            if 'blockX' in event:
                print(f"   Block Location: ({event.get('blockX', 'N/A')}, {event.get('blockY', 'N/A')})")
        elif event_type == 24:  # Stalls
            print(f"   Thrower: {event.get('thrower', 'N/A')} ({event.get('throwerX', 'N/A')}, {event.get('throwerY', 'N/A')})")
    
    print("\n" + "=" * 80)
    print(f"FINAL SCORE: {score_display}")
    print("=" * 80)
    print("Pipeline Complete!")
    print("=" * 80)

STEP 1: Cleaning and merging home/away events

Game: 2024-04-27-MTL-NY
Total events after cleaning: 750

STEP 2: Inserting missing turnovers

Original events: 750
Fixed events: 751
Synthetic turnovers added: 1

STEP 3: All Fixed Events (Chronological Order)

1. Starting O
   Team: royal
   Score: empire: 0 - royal: 0

2. Starting D
   Team: empire
   Score: empire: 0 - royal: 0

3. Pull (Inbounds)
   Team: royal
   Score: empire: 0 - royal: 0

4. Pass
   Team: empire
   Score: empire: 0 - royal: 0
   Thrower: skeegan (22.64, 7.09)
   Receiver: srueschem (4.63, 9.4)

5. Pass
   Team: empire
   Score: empire: 0 - royal: 0
   Thrower: srueschem (4.63, 9.4)
   Receiver: ochartock (-13.51, 18.61)

6. Throwaway
   Team: empire
   Score: empire: 0 - royal: 0
   Thrower: ochartock (-13.51, 18.61)
   Turnover Location: (-23.4, 38.31)

7. Defender Credit (Block)
   Team: royal
   Score: empire: 0 - royal: 0
   Defender: ctremblay

8. Pass
   Team: royal
   Score: empire: 0 - royal: 0
   Thrower:

In [15]:
# Run this to see what fields each event type actually has
cleaned_data = clean_data(["2024-04-27-MTL-NY"])
for game_id, events in cleaned_data.items():
    fixed_events = insert_missing_turnovers(events, seed=42)
    
    # Group by event type to see what fields exist
    from collections import defaultdict
    by_type = defaultdict(list)
    for event in fixed_events:
        by_type[event['type']].append(event)
    
    # Print sample of each type
    for event_type, events_of_type in sorted(by_type.items()):
        print(f"\nType {event_type}: {len(events_of_type)} events")
        print(f"Sample: {events_of_type[0].keys()}")


Type 1: 37 events
Sample: dict_keys(['type', 'line', 'time', 'team', 'year', 'gameID'])

Type 2: 37 events
Sample: dict_keys(['type', 'line', 'time', 'team', 'year', 'gameID'])

Type 3: 6 events
Sample: dict_keys(['type', 'line', 'time', 'team', 'year', 'gameID'])

Type 5: 6 events
Sample: dict_keys(['type', 'line', 'time', 'team', 'year', 'gameID'])

Type 7: 31 events
Sample: dict_keys(['type', 'puller', 'pullX', 'pullY', 'pullMs', 'team', 'year', 'gameID'])

Type 8: 6 events
Sample: dict_keys(['type', 'puller', 'team', 'year', 'gameID'])

Type 11: 29 events
Sample: dict_keys(['type', 'defender', 'team', 'year', 'gameID'])

Type 13: 19 events
Sample: dict_keys(['type', 'team', 'year', 'gameID'])

Type 14: 1 events
Sample: dict_keys(['type', 'team', 'year', 'gameID'])

Type 15: 34 events
Sample: dict_keys(['type', 'team', 'year', 'gameID'])

Type 16: 8 events
Sample: dict_keys(['type', 'team', 'year', 'gameID'])

Type 17: 8 events
Sample: dict_keys(['type', 'team', 'year', 'gameID'])
